# CILAS grain size processing: .MES files to plots

### <div style="text-align: right"> A.A. Lehrmann, October 2026 </div>

This notebook takes a folder of CILAS 1180 `.MES` files from one core, converts each to a CSV, re-bins each distribution to `Histogram (%) q3`, and plots them: one sample, the whole core coloured by depth, a depth package, and D-values and Wentworth fractions down core.

It replaces `mestocsv_Grain_size_calculations_plots.ipynb` and follows the same steps and the same `Histogram (%) q3` calculation, broken into separate cells. Changes from the old version:

* **The conversion keeps every size class and every header line.** The old script started one class late (dropping 0.01–0.04 µm), and its `Date` column actually read a grain size; the real date is header line 29.
* **The re-binning includes the finest classes.** The old loop started at the third row, so the first classes were left at zero.
* **Settings are asked for as the cells run**, and pasted paths are converted between Windows and Mac/Linux forms automatically.

### Before you begin

1. NEVER edit raw data. Do not delete .MES files. The notebook only reads them.
2. Put the core's `.MES` files in one folder. Filenames should end in the depth range, e.g. `NBP2002_KC67_30-32.MES`; the last `a-b` in the name is read as top-bottom depth in cm.
3. Run the cells top to bottom. Step 0c asks for the folders and core name, and Steps 3a, 3c and 3d ask which samples to plot. Press Enter to accept the default shown in [brackets].
4. Paste paths however you have them: with or without quotation marks, Windows style (`D:\Anuska gs\KC67`) or Mac/Linux style (`/Users/...`). If a path doesn't exist as typed, the notebook tries the other operating systems' forms of it before asking again (Step 0b).
5. Each step writes into its own subfolder of the output folder.

You'll need `numpy`, `pandas`, `matplotlib` and `cmcrameri` (`pip install cmcrameri`).

## Step 0a: Shared tools

Imports and the tools later steps use: depth parsing from filenames, the D-value calculation for the conversion check (Step 1c), and the figure style. Nothing to edit here.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import cmcrameri.cm as cmc

# ---- depth from filename: the LAST "a-b" number range in the name -----------
DEPTH_RE = re.compile(r"(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)")
def depth_from_name(name):
    m = DEPTH_RE.findall(os.path.basename(name))
    if not m:
        return np.nan, np.nan, np.nan
    a, b = sorted(float(x) for x in m[-1])
    return a, b, 0.5 * (a + b)

# ---- cumulative-curve tools for the Step 1c check ---------------------------
def edges_cum(d):
    # class limits and cumulative % finer at each limit, starting at 0 %
    return (np.r_[d["size_lower_um"].iloc[0], d["size_upper_um"].to_numpy(float)],
            np.r_[0.0, d["cum_pct"].to_numpy(float)])

def d_value(edges, cum, p):
    # size (µm) at p % finer
    return float(np.exp(np.interp(p, cum, np.log(edges))))

# ---- figure style -----------------------------------------------------------
CILAS_COLOR = cmc.batlow(0.10)
INK, INK2, RULE = "#0b0b0b", "#52514e", "#c9c8c3"
WENTWORTH = [(4, "clay | silt"), (63, "silt | sand"), (2000, "sand | gravel")]

def style_size_axis(ax, x_lo=0.01, x_hi=2500, label=True):
    ax.set_xscale("log"); ax.set_xlim(x_lo, x_hi)
    ticks = [t for t in (0.01, 0.1, 1, 10, 100, 1000) if x_lo <= t <= x_hi]
    ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks])
    for b, name in WENTWORTH:
        if x_lo < b < x_hi:
            ax.axvline(b, color=INK2, lw=1, ls=(0, (4, 3)), zorder=1)
            ax.annotate(name, (b, 1), xycoords=("data", "axes fraction"), xytext=(0, 4),
                        textcoords="offset points", ha="center", fontsize=8.5, color=INK2)
    ax.grid(color=RULE, lw=0.6); ax.set_axisbelow(True)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): ax.spines[sp].set_color(RULE)
    ax.tick_params(colors=INK2, labelsize=9)
    if label:
        ax.set_xlabel("Grain size (µm)", fontsize=10, color=INK)

## Step 0b: Path and input handling

Defines how the prompts in Step 0c (and Steps 3a, 3c, 3d) read what you type. Nothing to edit here.

**Paths.** Whatever you paste is cleaned first: surrounding quotation marks (as Windows "Copy as path" adds), spaces, `~` and `file://` are handled. If the path still doesn't exist on this computer, these forms are tried in turn, and the first that exists is used:

* backslashes and forward slashes swapped (`D:\KC67` ↔ `D:/KC67`);
* a Windows drive path as seen from WSL or Git Bash (`D:\KC67` → `/mnt/d/KC67`, `/d/KC67`), and the reverse on Windows (`/mnt/d/KC67` → `D:\KC67`);
* a Windows drive path on a Mac or Linux machine, where the same drive is mounted under `/Volumes/<drive name>/` (Mac) or `/media/<user>/<drive name>/`, `/run/media/<user>/<drive name>/` or `/mnt/<drive name>/` (Linux). Every mounted drive is checked for the rest of the path;
* a network path (`\\server\share\KC67`) as `/Volumes/share/KC67` (Mac) or `/mnt/share/KC67` (Linux).

The notebook prints the path it settled on whenever that differs from what you typed. If none of the forms exists, it lists what it tried and asks again.

**The output folder** doesn't have to exist yet. The same forms are tried, and the first whose parent folder exists is created. If no parent exists anywhere, you're asked again, so a Windows path on a Mac never turns into a folder with backslashes in its name.

**Other prompts** show a default in [brackets]; press Enter to accept it.

In [ ]:
_MOUNT_GLOBS = ["/Volumes/*", "/media/*/*", "/run/media/*/*", "/mnt/*"]

def _clean_path(raw):
    s = str(raw).strip().strip('"').strip("'").strip()
    if s.lower().startswith("file://"):
        from urllib.parse import unquote
        s = unquote(s[7:])
        if re.match(r"^/[A-Za-z]:", s):          # file:///C:/... on Windows
            s = s[1:]
    return os.path.expandvars(os.path.expanduser(s))

def path_candidates(raw):
    # every form this path might take on Windows, macOS, Linux, WSL or Git Bash, most likely first
    s = _clean_path(raw)
    fwd = s.replace("\\", "/")
    out = [s, fwd]
    drive = re.match(r"^([A-Za-z]):/*(.*)$", fwd)               # D:/rest
    unc = re.match(r"^//+([^/]+)/([^/]+)/?(.*)$", fwd)          # //server/share/rest
    if drive:
        letter, rest = drive.group(1), drive.group(2)
        out += [f"/mnt/{letter.lower()}/{rest}", f"/{letter.lower()}/{rest}"]
        for root in _MOUNT_GLOBS:                                # same drive mounted on Mac/Linux
            out += [os.path.join(m, rest) for m in sorted(glob.glob(root)) if os.path.isdir(m)]
    if unc:
        share, rest = unc.group(2), unc.group(3)
        out += [f"/Volumes/{share}/{rest}", f"/mnt/{share}/{rest}"]
    if os.name == "nt":
        m = re.match(r"^/(?:mnt/|cygdrive/)?([A-Za-z])(?:/(.*))?$", fwd)
        if m:
            out.append(f"{m.group(1).upper()}:\\" + (m.group(2) or "").replace("/", "\\"))
        out.append(s.replace("/", "\\"))
    seen, uniq = set(), []
    for c in out:
        c = os.path.normpath(c) if c else c
        if c and c not in seen:
            seen.add(c); uniq.append(c)
    return uniq

def resolve_dir(raw, must_exist=True):
    cands = path_candidates(raw)
    for c in cands:
        if os.path.isdir(c):
            return os.path.abspath(c), cands
    if not must_exist:
        for c in cands:
            parent = os.path.dirname(os.path.abspath(c))
            looks_windows = os.name != "nt" and ("\\" in c or re.match(r"^[A-Za-z]:", c))
            if os.path.isdir(parent) and not looks_windows:
                os.makedirs(c, exist_ok=True)
                return os.path.abspath(c), cands
    return None, cands

def ask_dir(prompt, must_exist=True, check=None, tries=5):
    for _ in range(tries):
        raw = input(f"{prompt}: ")
        path, cands = resolve_dir(raw, must_exist)
        if path is None:
            print("  Couldn't find that folder. Tried:", *cands, sep="\n    ")
            continue
        if os.path.normpath(_clean_path(raw)) != path:
            print(f"  Using: {path}")
        problem = check(path) if check else None
        if problem:
            print(f"  {problem}")
            continue
        return path
    raise SystemExit("No usable folder entered; run this cell again.")

def ask(prompt, default=None, cast=str):
    while True:
        shown = f" [{default}]" if default is not None else ""
        raw = input(f"{prompt}{shown}: ").strip()
        if not raw and default is not None:
            raw = default if isinstance(default, str) else None
            if raw is None:
                return default
        try:
            return cast(raw)
        except ValueError:
            print("  Couldn't read that; try again.")

## Step 0c: Enter your settings

Answer the prompts:

1. **Folder of `.MES` files** for one core. Must contain at least one `.MES` file.
2. **Output folder.** Everything this notebook writes goes here, in numbered subfolders (`01_csv`, `02_processed`, `03_plots`). Created if it doesn't exist.
3. **Core name**, used in plot titles and file names. The default is taken from the first file name with its depth range removed.

The cell then lists the `.MES` files it found, in depth order, with the depths it read from their names.

In [ ]:
def _has_mes(path):
    n = sum(f.lower().endswith(".mes") for f in os.listdir(path))
    return None if n else "No .MES files in that folder; try again."

MES_DIR = ask_dir("Folder of .MES files", must_exist=True, check=_has_mes)
OUT_DIR = ask_dir("Output folder (created if needed)", must_exist=False)

mes_files = sorted((f for f in os.listdir(MES_DIR) if f.lower().endswith(".mes")),
                   key=lambda f: (np.inf if np.isnan(depth_from_name(f)[0]) else depth_from_name(f)[0], f))
_default_core = re.sub(r"[_\s-]*\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?[^/\\]*$", "", os.path.splitext(mes_files[0])[0]) or "core"
CORE_NAME = ask("Core name", default=_default_core)

CSV_DIR   = os.path.join(OUT_DIR, "01_csv")
PROC_DIR  = os.path.join(OUT_DIR, "02_processed")
PLOT_DIR  = os.path.join(OUT_DIR, "03_plots")
for d in (CSV_DIR, PROC_DIR, PLOT_DIR):
    os.makedirs(d, exist_ok=True)
CORE_TAG = re.sub(r"[^A-Za-z0-9._-]+", "_", CORE_NAME).strip("_")

print(f"\n.MES folder : {MES_DIR}\nOutput      : {OUT_DIR}\nCore        : {CORE_NAME}")
print(f"\n{len(mes_files)} .MES files:")
for f in mes_files:
    a, b, mid = depth_from_name(f)
    print(f"  {f:40s} depth {a:g}-{b:g} cm" if np.isfinite(a) else f"  {f:40s} (no depth range in name)")

## Step 1a: How a .MES file is read

The `.MES` file is plain text, one value per line, with no field names:

* **Lines 1 to 43, header.** Every line is kept as its own column, `MES_line_01` to `MES_line_43`. Lines whose meaning has been checked carry a tag in the column name. Line 18 is CILAS's own D50, and lines 19/20 and 21/22 are "10 → D10" and "90 → D90" (all three match the D-values calculated from the curve). Line 23 matches the specific surface area calculated from the curve (cm²/cm³), so it is tagged "inferred". Line 26 is the operator and line 29 the measurement date and time.
* **Size block: 101 class limits** in µm (0.01 to 2500).
* **Cumulative block: 100 cumulative undersize fractions** (0 to 1). Value *k* is the fraction finer than size limit *k + 1*, so class *k* runs from limit *k* to limit *k + 1*.

The two blocks are found by their content, not by fixed line numbers: the cumulative block is the run of values ≤ 1 at the end of the file, and the size block is the 101 lines before it. A file that doesn't fit this pattern stops with an error instead of being misread.

This cell only defines the reader. Step 1b runs it.

In [ ]:
MES_LINE_TAGS = {
    1: "file format version", 2: "instrument model", 5: "sample name",
    18: "D50 um", 19: "percentile (10)", 20: "D10 um", 21: "percentile (90)", 22: "D90 um",
    23: "specific surface cm2 per cm3 (inferred)", 26: "operator",
    29: "measurement date time", 31: "software version",
}

def _is_float(s):
    try:
        float(s); return True
    except ValueError:
        return False

def read_mes(path):
    with open(path, "r", encoding="latin-1") as fh:
        lines = [ln.rstrip("\r\n") for ln in fh]
    while lines and not lines[-1].strip():
        lines.pop()

    # cumulative block: walk back from the end while values are <= 1 and non-increasing going up
    j, prev = len(lines) - 1, np.inf
    while j >= 0 and _is_float(lines[j]) and float(lines[j]) <= 1.0 and float(lines[j]) <= prev:
        prev = float(lines[j]); j -= 1
    cum_start = j + 1
    n_cls = len(lines) - cum_start
    size_start = cum_start - (n_cls + 1)

    sizes = np.array([float(x) for x in lines[size_start:cum_start]])
    cumf = np.array([float(x) for x in lines[cum_start:]])
    if size_start < 1 or np.any(np.diff(sizes) <= 0) or abs(cumf[-1] - 1) > 1e-3:
        raise ValueError(f"{path}: size/cumulative blocks not where expected; check the file")

    meta = {}
    for i, txt in enumerate(lines[:size_start], start=1):
        tag = MES_LINE_TAGS.get(i)
        meta[f"MES_line_{i:02d}" + (f" [{tag}]" if tag else "")] = txt.strip()
    return sizes, cumf, meta

## Step 1b: Convert every .MES file to a CSV

One CSV per `.MES` file, one row per size class, saved as `01_csv/<file name>.csv`. Columns:

* **Identification:** `instrument` (CILAS), `source_file`, `sample_name` (header line 5), `top_cm`, `bottom_cm`, `mid_cm` (from the filename).
* **Distribution:** `class_index`, `size_lower_um`, `size_upper_um`, `um` (= upper limit, same convention as the old notebook), `freq_pct` (% volume in the class), `cum_pct` (cumulative % finer than `size_upper_um`), and `cum_fraction` (the raw 0–1 value from the file).
* **Header:** every header line as a `MES_line_XX` column, repeated on every row so each CSV stands on its own.

In [ ]:
converted = {}
for f in mes_files:
    sizes, cumf, meta = read_mes(os.path.join(MES_DIR, f))
    top, bot, mid = depth_from_name(f)
    cum_pct = cumf * 100
    d = pd.DataFrame({
        "instrument": "CILAS", "source_file": f, "sample_name": meta["MES_line_05 [sample name]"],
        "top_cm": top, "bottom_cm": bot, "mid_cm": mid,
        "class_index": np.arange(1, len(cumf) + 1),
        "size_lower_um": sizes[:-1], "size_upper_um": sizes[1:], "um": sizes[1:],
        "freq_pct": np.diff(np.r_[0.0, cum_pct]), "cum_pct": cum_pct, "cum_fraction": cumf,
    })
    d = pd.concat([d, pd.DataFrame([meta] * len(d))], axis=1)
    out = os.path.join(CSV_DIR, os.path.splitext(f)[0] + ".csv")
    d.to_csv(out, index=False)
    converted[f] = d
print(f"Converted {len(converted)} files to {CSV_DIR}")

## Step 1c: Check the conversion against CILAS's own numbers

This is the replacement for the old "double check the D50" cell. For every converted file it:

1. checks the classes sum to 100% and the cumulative curve ends at 100%;
2. recalculates D10, D50 and D90 from the CSV's cumulative curve (interpolating between the CILAS size limits on a log scale) and compares them with the values CILAS wrote in the header (lines 20, 18 and 22).

Differences of a few hundredths of a µm are rounding. Anything larger means a file was read wrongly, or its header and data don't belong together. Files are flagged when D50 is off by more than 2%. The table is saved as `01_csv/conversion_check.csv`.

In [ ]:
rows = []
for f, d in converted.items():
    e, c = edges_cum(d)
    row = {"file": f, "mid_cm": d["mid_cm"].iloc[0], "freq_sum": round(d["freq_pct"].sum(), 3),
           "cum_end": round(c[-1], 3)}
    for p, line in ((10, "MES_line_20 [D10 um]"), (50, "MES_line_18 [D50 um]"), (90, "MES_line_22 [D90 um]")):
        row[f"D{p}_calc"] = round(d_value(e, c, p), 3)
        row[f"D{p}_cilas"] = float(d[line].iloc[0]) if _is_float(str(d[line].iloc[0])) else np.nan
    row["D50_diff_pct"] = round(100 * (row["D50_calc"] / row["D50_cilas"] - 1), 2)
    row["flag"] = "CHECK" if abs(row["D50_diff_pct"]) > 2 or abs(row["freq_sum"] - 100) > 0.5 else ""
    rows.append(row)
check = pd.DataFrame(rows)
check.to_csv(os.path.join(CSV_DIR, "conversion_check.csv"), index=False)
display(check)
print(f"{(check['flag'] == 'CHECK').sum()} of {len(check)} files flagged")

## Step 2: Re-bin to `Histogram (%) q3`

This is the calculation from the old notebook, applied to every converted file. For each size class *i* (one row of the CSV, with `um` its upper size limit):

1. **`Cumulative (%) Q3`**: the cumulative % of the sample finer than `um` (the `cum_pct` column).
2. **`Computation`**: the change in cumulative % across the class, divided by the class width on a log scale:

   $$\mathrm{Computation}_i = \frac{Q3_i - Q3_{i-1}}{\ln\left(\mathrm{um}_i \,/\, \mathrm{um}_{i-1}\right)}$$

   For the first class, $Q3_{i-1} = 0$ and $\mathrm{um}_{i-1}$ is its lower limit (0.01 µm), so all 100 classes are included.
3. **`Total Computation`**: the sum of `Computation` over all classes.
4. **`Histogram (%) q3`**: each class's `Computation` as a percentage of the total, so the values sum to 100:

   $$\mathrm{Histogram\ (\%)\ q3}_i = \frac{\mathrm{Computation}_i}{\mathrm{Total\ Computation}} \times 100$$

Each file is saved as `02_processed/processed_<file>.csv`, with every column from Step 1b plus these four, the same names and file naming as the old notebook. All the plots in Step 3 use `Histogram (%) q3`.

In [ ]:
processed = {}
for f, d in converted.items():
    p = d.copy()
    q3 = p["cum_pct"].to_numpy(float)
    um = p["um"].to_numpy(float)
    prev_q3 = np.r_[0.0, q3[:-1]]
    prev_um = np.r_[p["size_lower_um"].iloc[0], um[:-1]]
    p["Cumulative (%) Q3"] = q3
    p["Computation"] = (q3 - prev_q3) / np.log(um / prev_um)
    p["Total Computation"] = p["Computation"].sum()
    p["Histogram (%) q3"] = p["Computation"] / p["Total Computation"] * 100
    p.to_csv(os.path.join(PROC_DIR, "processed_" + os.path.splitext(f)[0] + ".csv"), index=False)
    processed[f] = p

print(f"Saved {len(processed)} processed CSVs to {PROC_DIR}")
display(pd.DataFrame({"file": list(processed),
                      "sum of Histogram (%) q3": [round(p["Histogram (%) q3"].sum(), 3) for p in processed.values()],
                      "Total Computation": [round(p["Total Computation"].iloc[0], 2) for p in processed.values()]}))

## Step 3a: One sample

The replacement for the old single-sample plot. The two curves are drawn as two panels sharing the grain size axis, instead of one plot with two y-axes:

* **Top:** `Cumulative (%) Q3` against `um`.
* **Bottom:** `Histogram (%) q3` against `um`.

The cell lists the files with numbers and asks which one to plot: type its number or its name, or press Enter for the first. Saved as `03_plots/sample_<file>.png`.

In [ ]:
for i, name in enumerate(mes_files, 1):
    print(f"  {i:3d}  {name}")
def _pick(raw):
    if raw.isdigit() and 1 <= int(raw) <= len(mes_files):
        return mes_files[int(raw) - 1]
    match = [m for m in mes_files if m.lower() == raw.lower() or os.path.splitext(m)[0].lower() == raw.lower()]
    if not match:
        raise ValueError
    return match[0]
f = ask("Sample to plot (number or file name)", default=mes_files[0], cast=_pick)
p = processed[f]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})
ax1.plot(p["um"], p["Cumulative (%) Q3"], color=CILAS_COLOR, lw=2.2)
style_size_axis(ax1, label=False)
ax1.set_ylim(0, 100); ax1.set_ylabel("Cumulative (%) Q3", fontsize=10, color=INK)
ax1.set_title(f"{p['sample_name'].iloc[0]}", loc="left", fontsize=12, color=INK, fontweight="bold", pad=22)

ax2.plot(p["um"], p["Histogram (%) q3"], color=CILAS_COLOR, lw=2.2)
style_size_axis(ax2)
ax2.set_ylim(bottom=0); ax2.set_ylabel("Volume percent (%)", fontsize=10, color=INK)
fig.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, "sample_" + os.path.splitext(f)[0] + ".png"), dpi=200, bbox_inches="tight")
plt.show()

## Step 3b: Every sample in the core, coloured by depth

The replacement for the old "one core" plot: `Histogram (%) q3` against `um` for every file, each line coloured by its mid-depth with the batlow colour map (shallow = dark blue, deep = pink), and a colour bar for depth. Files without a depth range in their name are drawn in grey. When there are 24 files or fewer, a legend of file names goes under the plot, as in the old version.

Saved as `03_plots/<core>_all_samples.png`.

In [ ]:
def plot_depth_coloured(files, title, out_png):
    mids = np.array([processed[f]["mid_cm"].iloc[0] for f in files], float)
    fin = mids[np.isfinite(mids)]
    lo_d, hi_d = (fin.min(), fin.max()) if fin.size else (0.0, 1.0)
    if lo_d == hi_d:
        lo_d, hi_d = lo_d - 0.5, hi_d + 0.5
    norm = mpl.colors.Normalize(vmin=lo_d, vmax=hi_d)

    fig, ax = plt.subplots(figsize=(12, 6))
    for f, mid in zip(files, mids):
        p = processed[f]
        color = cmc.batlow(norm(mid)) if np.isfinite(mid) else (0.6, 0.6, 0.6, 1.0)
        ax.plot(p["um"], p["Histogram (%) q3"], color=color, lw=1.8, label=os.path.splitext(f)[0])
    style_size_axis(ax)
    ax.set_ylim(bottom=0); ax.set_ylabel("Volume percent (%)", fontsize=10, color=INK)
    ax.set_title(title, loc="left", fontsize=12, color=INK, fontweight="bold", pad=22)
    sm = mpl.cm.ScalarMappable(cmap=cmc.batlow, norm=norm); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, pad=0.015)
    cb.set_label("Mid-depth of interval (cm)", fontsize=10, color=INK)
    cb.ax.invert_yaxis()                        # shallow at the top, like a core
    if len(files) <= 24:
        ax.legend(frameon=False, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
    fig.tight_layout()
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()

plot_depth_coloured(mes_files, f"Grain size distributions, core {CORE_NAME} ({len(mes_files)} samples)",
                    os.path.join(PLOT_DIR, f"{CORE_TAG}_all_samples.png"))

## Step 3c: A depth package within the core

Same plot as 3b for a subset of samples, chosen by the TOP depth in each filename. The prompt takes:

* a range, e.g. `0-40`, to keep samples whose top depth is in that range (cm);
* a list, e.g. `0, 10, 24.5`, to keep samples whose top depth matches any of those values (you're then asked for a tolerance, default 0.02 cm);
* `all`, or Enter, to keep everything.

The cell prints which files it kept. Saved as `03_plots/<core>_package_<range>.png`.

In [ ]:
def _package(raw):
    raw = raw.strip().lower()
    if raw in ("", "all"):
        return None, None
    if "," in raw:
        return None, [float(x) for x in raw.split(",") if x.strip()]
    m = re.match(r"^\s*(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)\s*$", raw)
    if m:
        a, b = sorted(float(x) for x in m.groups())
        return (a, b), None
    return None, [float(raw)]                      # a single top depth

PACKAGE_RANGE, PACKAGE_TOPS = ask("Top depths: range like 0-40, list like 0,10,24.5, or all",
                                  default="all", cast=_package)
TOP_TOL = ask("Tolerance for matching top depths (cm)", default=0.02, cast=float) if PACKAGE_TOPS else 0.02

def keep(f):
    top = processed[f]["top_cm"].iloc[0]
    if not np.isfinite(top):
        return False
    if PACKAGE_RANGE is not None:
        return PACKAGE_RANGE[0] <= top <= PACKAGE_RANGE[1]
    if PACKAGE_TOPS is not None:
        return any(abs(top - t) <= TOP_TOL for t in PACKAGE_TOPS)
    return True

package = [f for f in mes_files if keep(f)]
print("Selected:", *package, sep="\n  ")
if not package:
    raise SystemExit("No files matched the package selection.")

if PACKAGE_RANGE is not None:
    note, tag = f"top depths {PACKAGE_RANGE[0]:g}-{PACKAGE_RANGE[1]:g} cm", f"{PACKAGE_RANGE[0]:g}-{PACKAGE_RANGE[1]:g}cm"
elif PACKAGE_TOPS is not None:
    note, tag = f"top depths {', '.join(f'{t:g}' for t in PACKAGE_TOPS)} cm", "list"
else:
    note, tag = "all samples", "all"
plot_depth_coloured(package, f"Grain size distributions, core {CORE_NAME}: {note}",
                    os.path.join(PLOT_DIR, f"{CORE_TAG}_package_{tag}.png"))

## Step 3d: D-values and Wentworth fractions down core

The replacement for the old D50-vs-depth cell, with the Wentworth fractions added. For every sample, from its `Histogram (%) q3`:

* **D10, D30, D50, D90 (µm)**, as in the old notebook: the running total of `Histogram (%) q3` (scaled to end at 100) is the cumulative curve, and each D-value is the `um` where it reaches 10, 30, 50 or 90%, interpolated linearly between rows.
* **Clay, silt, sand and gravel (%)**, as in the Wentworth notebook: `Histogram (%) q3` summed over the rows with `um` < 4 µm (clay), 4 to < 63 µm (silt), 63 to < 2000 µm (sand) and ≥ 2000 µm (gravel).
* **Mode (µm)**: the `um` of the row with the highest `Histogram (%) q3`.

The table is saved as `<core>_downcore_summary.csv`. The figure shows D50 with a D10–D90 band (left) and the stacked clay, silt and sand fractions (right; gravel, if any, is added to sand) against depth, depth increasing downwards. Samples are plotted at their top depth, as in the old D50 plot.

The prompt asks whether to use every sample or only the Step 3c selection.

In [ ]:
def _all_or_package(raw):
    raw = raw.strip().lower()
    if raw not in ("all", "package"):
        raise ValueError
    return raw
_which = ask("Use all samples or the Step 3c package? (all/package)", default="all", cast=_all_or_package)
DEPTH_FILES = package if _which == "package" else mes_files

rows = []
for f in DEPTH_FILES:
    p = processed[f]
    um, H = p["um"].to_numpy(float), p["Histogram (%) q3"].to_numpy(float)
    cdf = np.cumsum(H) / H.sum() * 100
    rows.append({"file": f, "sample": p["sample_name"].iloc[0], "top_cm": p["top_cm"].iloc[0],
                 "bottom_cm": p["bottom_cm"].iloc[0], "mid_cm": p["mid_cm"].iloc[0],
                 **{f"D{q}_um": round(float(np.interp(q, cdf, um)), 3) for q in (10, 30, 50, 90)},
                 "mode_um": round(float(um[H.argmax()]), 2),
                 "clay_pct": round(float(H[um < 4].sum()), 2),
                 "silt_pct": round(float(H[(um >= 4) & (um < 63)].sum()), 2),
                 "sand_pct": round(float(H[(um >= 63) & (um < 2000)].sum()), 2),
                 "gravel_pct": round(float(H[um >= 2000].sum()), 2)})
down = pd.DataFrame(rows).sort_values(["top_cm", "file"]).reset_index(drop=True)
down.to_csv(os.path.join(OUT_DIR, f"{CORE_TAG}_downcore_summary.csv"), index=False)
display(down)

z = down["top_cm"].to_numpy()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.5, 7), sharey=True, gridspec_kw={"width_ratios": [1, 1.1]})
a1.fill_betweenx(z, down["D10_um"], down["D90_um"], color=CILAS_COLOR, alpha=0.18, lw=0, label="D10-D90")
a1.plot(down["D50_um"], z, color=CILAS_COLOR, lw=2, marker="o", ms=4, label="D50")
a1.set_xscale("log"); a1.set_xlabel("Grain size (µm)", fontsize=10, color=INK)
a1.xaxis.set_major_locator(mpl.ticker.FixedLocator([0.2, 0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500]))
a1.xaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda v, _: f"{v:g}"))
a1.xaxis.set_minor_formatter(mpl.ticker.NullFormatter())
a1.set_ylabel("Depth (cm)", fontsize=10, color=INK)
a1.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)

FRACTION_COLORS = {"clay_pct": "#c8a27a",      # light brown
                   "silt_pct": "#d4d4d4",      # light grey
                   "sand_pct": "#f7e99a"}      # pastel yellow
left = np.zeros(len(down))
for col, name in (("clay_pct", "Clay < 4 µm"), ("silt_pct", "Silt 4-63 µm"), ("sand_pct", "Sand > 63 µm")):
    vals = down[col].to_numpy() + (down["gravel_pct"].to_numpy() if col == "sand_pct" else 0)
    a2.fill_betweenx(z, left, left + vals, color=FRACTION_COLORS[col], lw=0, label=name)
    left = left + vals
for edge in np.cumsum([down["clay_pct"].to_numpy(), down["silt_pct"].to_numpy()], axis=0):
    a2.plot(edge, z, color=INK2, lw=0.6)          # thin boundary lines between fractions
a2.set_xlim(0, 100); a2.set_xlabel("Volume (%)", fontsize=10, color=INK)
a2.legend(frameon=False, fontsize=9, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)

for a in (a1, a2):
    a.grid(color=RULE, lw=0.6); a.set_axisbelow(True)
    for sp in ("top", "right"): a.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): a.spines[sp].set_color(RULE)
    a.tick_params(colors=INK2, labelsize=9)
a1.invert_yaxis()
fig.suptitle(f"Core {CORE_NAME}: grain size down core", x=0.06, ha="left", fontsize=12,
             color=INK, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(PLOT_DIR, f"{CORE_TAG}_downcore.png"), dpi=300, bbox_inches="tight")
plt.show()